# Tweezer Experiment - Clean Walkthrough

This notebook keeps the structure of the original day-to-day experiment showcase, but removes the repeated experiment blocks.

The repeated sections in the source notebook are mostly the same workflow with small parameter changes:
- camera and experiment setup
- phasemask generation
- baseline detection and loading statistics
- rearrangement node initialisation and acquisition loops
- image inspection and animation

This version keeps one clean template for each step, plus a compact configuration list for the repeated runs.

In [ ]:
import asyncio
from time import sleep

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from scipy.ndimage import median_filter
from scipy.stats import norm

import pytweezer.phasemask as pm
import pytweezer.communication as com
from pytweezer.experiment import motmaster_client
from pytweezer.analysis import analysis as pyt
from pytweezer.drivers.imagemX2 import ImagEMX2Camera, ImagEMX2CameraClient
from pytweezer.drivers import anapico

In [ ]:
# Core experiment objects used throughout the notebook.
texp = pyt.TweezerExperimentAnalysis(day='03', month='06Jun', year='2026')
exp = motmaster_client.MotMasterClient('Rb')
local_cam: ImagEMX2Camera = ImagEMX2Camera()

# Camera configuration from the original notebook.
local_cam.set_trigger_source('ext')
local_cam.set_external_exposure_mode()
local_cam.enable_em_gain(True)
local_cam.enable_direct_em_gain(True)
local_cam.set_sensitivity(1200)
local_cam.timeout = 60 * 20
X0, Y0, WIDTH, HEIGHT = 124, 170, 256, 256
local_cam.set_roi(X0, WIDTH, Y0, HEIGHT)

# Phasemask generator and SLM client.
dx, dy = 8.0, 4.0
PM = pm.OptimisationBasedPhasemaskGeneratorGPU(
    wavelength_um=0.852,
    focal_length_mm=17.3,
    slm_pitch_um=17,
    slm_res=(1024, 1024),
    input_beam_waist_mm=16,
    fresnel_f_mm=1072,
    blaze_dx_dy_um=(40 + dx, -8 + dy),
    zernike_coeff_dict={5: 1.195, 6: 0.725, 7: 0.970, 8: 0.478, 9: -1.091, 10: 0.303, 11: 0.021, 12: 0.072, 13: 0.049},
)
SLM = com.SLMClient()

## Reusable helpers

The original notebook redefined the same analysis and acquisition logic several times. These helpers keep that logic in one place.

In [ ]:
gauss2d = lambda x, y, A, x0, y0, sx, sy, offset: (A * np.exp(-(((x - x0) ** 2) / (2 * sx ** 2) + ((y - y0) ** 2) / (2 * sy ** 2)))) + offset

def build_phasemask(array_size, spacing_um, angle_deg=2.5, randomness=1.0, weights=None):
    if weights is None:
        weights = np.ones(array_size)
    w_n, theta_n, x_n, y_n, array_shape = PM.generate_weighted_array(
        weights, spacing_um, init_phase_randomness=randomness, angle_deg=angle_deg
    )
    return w_n, theta_n, x_n, y_n, array_shape

def upload_phasemask(weight_matrix, spacing_um, fit_params, max_iter=2000, damping=0.5):
    A, x0, y0, sx, sy, offset = fit_params
    w_n, theta_n, x_n, y_n, array_shape = build_phasemask(weight_matrix.shape, spacing_um, weights=weight_matrix)
    w_n = (weight_matrix * gauss2d(x_n.reshape(array_shape), y_n.reshape(array_shape), A, x0, y0, sx, sy, offset)).flatten()
    terms = [w_n, theta_n, x_n, y_n, array_shape]
    pm_array, terms, _ = PM.superposition_optimization(terms, max_iter=max_iter, damping=damping, verbose=True)
    pm_composite = PM.superimpose([pm_array, PM.fresnel, PM.blaze, PM.zernike])
    pm_composite_uint8 = PM.transform_phase_8bit(pm_composite).get()
    SLM.update_mask(pm_composite_uint8)
    return pm_array, terms, pm_composite_uint8

def detect_and_measure(imgs, array_shape, feature_size=10, window_size=3, detection_step=500, threshold_detection=True, threshold=None):
    filtered_imgs = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs]
    mean_img = np.mean(filtered_imgs, axis=0)
    grid_positions, detection_threshold = pyt.detect_trap_sites(mean_img, array_shape, detection_step=detection_step)
    pyt.visualize_results(mean_img, grid_positions, margin=50, window_size=window_size, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)
    stats_kwargs = dict(threshold_detection=threshold_detection, window_size=window_size, binning=60)
    if threshold is not None:
        stats_kwargs['threshold'] = threshold
    photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(
        filtered_imgs, grid_positions, array_shape, **stats_kwargs
    )
    return {
        'filtered_imgs': filtered_imgs,
        'mean_img': mean_img,
        'grid_positions': grid_positions,
        'detection_threshold': detection_threshold,
        'photon_rates': photon_rates,
        'loading_probabilities': loading_probabilities,
        'threshold': threshold,
        'fidelity': fidelity,
    }

def extract_survival_probability(imgs1, imgs2, grid_positions, array_shape, window_size=3):
    pr_1, _, thresh_1, _ = texp.get_array_loading_statistics(imgs1, grid_positions, array_shape, threshold_detection=True, window_size=window_size, binning=60, show_histogram=False, verbose=False)
    pr_2, _, thresh_2, _ = texp.get_array_loading_statistics(imgs2, grid_positions, array_shape, threshold_detection=True, window_size=window_size, binning=60, show_histogram=False, verbose=False)
    survival_fractions = []
    for i in range(pr_1.shape[0]):
        pr_mat_1 = pr_1[i]
        pr_mat_2 = pr_2[i]
        pr_mat_1_binary = (pr_mat_1 > thresh_1).astype(int)
        pr_mat_2_binary = (pr_mat_2 > thresh_2).astype(int)
        survival_matrix = pr_mat_1_binary * pr_mat_2_binary
        init_occupation = pr_mat_1_binary.sum()
        survival_count = survival_matrix.sum()
        survival_fraction = survival_count / init_occupation if init_occupation > 0 else 0
        survival_fractions.append(survival_fraction)
    return float(np.mean(survival_fractions))

def create_animation(frames, interval=200):
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(frames[0], cmap='magma', vmax=np.array(frames).max() * 0.6)
    plt.close(fig)

    def update(frame):
        im.set_data(frame)
        return [im]

    anim = FuncAnimation(fig, update, frames=frames, interval=interval, blit=True)
    return HTML(anim.to_jshtml())

## Clean workflow

This is the condensed path through the experiment. It keeps the key calibration stages and replaces the repeated `Exp 1` to `Exp 7` blocks with a single parameterized run list.

In [ ]:
# Example array definitions from the original notebook.
fit_params = [-2.14197604e+03, -1.11443850e+01 - dx, 2.09919600e+00 - dy, 2.95002910e+03, 2.64281503e+03, 2.14239505e+03]
spacing_um = 5.0

initial_weights = np.ones((16, 16))
final_weights = np.ones((10, 10))

pm_array1, terms1, pm_mask1 = upload_phasemask(initial_weights, spacing_um, fit_params)
pm_array2, terms2, pm_mask2 = upload_phasemask(final_weights, spacing_um, fit_params)

run_configs = [
    {'name': 'Exp 1', 'd0': 1.2, 'final_threshold': 0.5, 'threshold_detection': False},
    {'name': 'Exp 2', 'd0': 1.2, 'final_threshold': 0.5, 'threshold_detection': False},
    {'name': 'Exp 3', 'd0': 1.0, 'final_threshold': 0.5, 'threshold_detection': False},
    {'name': 'Exp 4', 'd0': 1.0, 'final_threshold': 0.5, 'threshold_detection': False},
    {'name': 'Exp 5', 'd0': 1.2, 'final_threshold': 0.5, 'threshold_detection': False},
    {'name': 'Exp 6', 'd0': 1.0, 'final_threshold': 1.0, 'threshold_detection': False},
    {'name': 'Exp 7', 'd0': 1.0, 'final_threshold': 1.2, 'threshold_detection': False},
]

async def initialise_rearrangement_cycle(run_config, terms1, terms2, grid_positions, threshold_counts, roi):
    return await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=run_config['d0'], fps=1 / (1e-6), roi=roi)

# Use the same acquisition loop for every run instead of duplicating the code block per experiment.
rearrangement_template = {
    'iterations': 100,
    'background_drop': 0,
    'sleep_s': 0.1,
}

In [ ]:
# Baseline scan example.
n_iterations = 100
exp.set_motmaster_experiment('RbTweezerBasic2026_2')
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)
local_cam.setup_acquisition('snap', n_iterations * 2)
local_cam.start_acquisition()
exp.start_motmaster_experiment()

imgs = local_cam.acquire_n_frames(n_iterations * 2)
imgs1, imgs2 = imgs[::2], imgs[1::2]
img_average = texp.tweezer_show_bg_subtracted(imgs1, imgs2, reg=[0, -1, 0, -1], cmap='gray', show=True, vmaxfactor=0.6, show_grid=True)

baseline = detect_and_measure(imgs1, [16, 16], feature_size=10, window_size=3, detection_step=100, threshold_detection=True)
grid_positions = baseline['grid_positions']
threshold_counts = baseline['threshold'] * 1000 * 80e-3 / 0.00483372 + 1828.38

print(f'Estimated threshold counts: {threshold_counts:.2f}')
print(f"Loaded filling fraction: {np.mean([np.sum(pr > baseline['threshold']) for pr in baseline['photon_rates']]) / 256:.2%}")

In [ ]:
# Rearrangement summary for one representative run.
# The original notebook repeated this block with only small changes in d0 and final-array threshold settings.

RN = None  # Instantiate pytweezer.communication.RearrangementNode in a live session.

# Example outline only. In practice, run one config at a time:
# await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.2, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])
# imgs = []
# for it in range(100):
#     rearrangement_task = asyncio.create_task(RN.arm_rearrangement())
#     sleep(0.1)
#     exp.start_motmaster_experiment({'BackgroundDrop': 0})
#     img0, img1, timings = await rearrangement_task
#     imgs.append([img0, img1])

# After the run, filter and inspect the two image streams.
# imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=10) for img in imgs1]
# imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=10) for img in imgs2]

## Synthetic Data Demo

This section uses the sample generator in `pytweezer.analysis.synthetic_tweezer_demo` so you can inspect the same analysis flow without connecting to hardware.

In [ ]:
from pytweezer.analysis.synthetic_tweezer_demo import SyntheticDemoConfig, build_demo_dataset

demo = build_demo_dataset(SyntheticDemoConfig(sample_frames=24, animation_frames=20))
signal_images = demo["signal_images"]
background_images = demo["background_images"]
frames = demo["frames"]

print(f"Synthetic loading threshold: {demo['threshold']:.3f}")
print(f"Synthetic detection fidelity: {demo['fidelity']:.3f}")
print(f"Mean loading probability: {demo['loading_probabilities'].mean():.3f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(signal_images[0], cmap='magma')
plt.title('Synthetic signal frame')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(demo["background_subtracted"], cmap='gray')
plt.title('Background-subtracted mean')
plt.axis('off')
plt.tight_layout()
plt.show()